In [ ]:
import logging
import torch
import gc
import warnings
from simpletransformers.ner import NERModel, NERArgs


def initialize_model(model_name, unique_labels, no_progress_bars=False, cleanup=True):
    """Intializes the CamemBERT model with specified name and all possible labels list.

    Args:
        model_name (str): Default Transformer model name or path to a directory containing Transformer model file
        unique_labels (List[str]): List of all possible unique labels model can predict
        no_progress_bars (bool, optional): Whether to output progress bars. Defaults to False.
        cleanup (bool, optional): Will clean up any existing model before reinitialization.

    Returns:
        simpletransformers.ner.NERModel: Initialized model
    """
    if cleanup and "model" in globals():
        del globals()["model"]
    if cleanup and "model" in locals():
        del locals()["model"]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Set up logging
    logger = logging.getLogger("simpletransformers.ner.ner_model")
    logger.setLevel(logging.ERROR)

    # Suppress specific warnings
    # warnings.filterwarnings("ignore", category=FutureWarning) # For warning message "FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated."
    warnings.filterwarnings(
        "ignore", category=UserWarning
    )  # For warnings like "UserWarning: <tag> seems not to be NE tag."

    # Configurations
    model_args = NERArgs()
    model_args.train_batch_size = 8
    model_args.evaluate_during_training = False
    model_args.learning_rate = 5e-5
    model_args.num_train_epochs = 10
    model_args.use_early_stopping = True
    model_args.use_cuda = torch.cuda.is_available()  # Use GPU if available
    model_args.save_eval_checkpoints = False
    model_args.save_model_every_epoch = False  # Takes a lot of storage space
    model_args.save_steps = -1
    model_args.overwrite_output_dir = True
    model_args.cache_dir = model_name + "/cache"
    model_args.best_model_dir = model_name + "/best_model"
    model_args.output_dir = model_name
    model_args.use_multiprocessing = False
    model_args.silent = no_progress_bars

    # Initialization
    model = NERModel("camembert", model_name, args=model_args, labels=unique_labels)
    return model

In [ ]:
import typing
import logging
import torch
import gc
import warnings
from transformers import AutoTokenizer, AutoConfig, AutoModelForTokenClassification


def initialize_model(
    model_name: str,
    unique_labels: typing.List[str],
    no_progress_bars: bool = False,
    cleanup: bool = True,
) -> typing.Dict[str, typing.Any]:
    """
    Initialise a token-classification model (for NER) using Hugging Face transformers.

    Args:
        model_name: Pretrained model name or local path (e.g. 'camembert-base').
        unique_labels: All possible label strings the model will predict.
        no_progress_bars: If True, reduces HF logging verbosity.
        cleanup: If True, attempts to free previous model memory before initialisation.

    Returns:
        A dict with keys:
        - 'model': AutoModelForTokenClassification
        - 'tokenizer': AutoTokenizer
        - 'config': AutoConfig
        - 'label2id': dict[str,int]
        - 'id2label': dict[int,str]
        - 'device': torch.device
        - 'training_args': small dict with training hyperparameters
    """
    if cleanup and "model" in globals():
        try:
            del globals()["model"]
        except Exception:
            pass

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Logging / warnings
    logging.getLogger("transformers").setLevel(logging.ERROR)
    if no_progress_bars:
        from transformers.utils import logging as hf_logging

        hf_logging.set_verbosity_error()
    warnings.filterwarnings("ignore", category=UserWarning)

    # Build label mappings expected by transformers config
    label2id = {label: i for i, label in enumerate(unique_labels)}
    id2label = {i: label for label, i in label2id.items()}

    # Load config, tokenizer and model
    config = AutoConfig.from_pretrained(
        model_name,
        num_labels=len(unique_labels),
        id2label=id2label,
        label2id=label2id,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = AutoModelForTokenClassification.from_pretrained(model_name, config=config)

    # Move model to device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    training_args = {
        "train_batch_size": 8,
        "learning_rate": 5e-5,
        "num_train_epochs": 10,
        "use_cuda": torch.cuda.is_available(),
    }

    return {
        "model": model,
        "tokenizer": tokenizer,
        "config": config,
        "label2id": label2id,
        "id2label": id2label,
        "device": device,
        "training_args": training_args,
    }